# Example use-case: One-zone chemical evolution model.
using persistent data and accessing previous convolution results

In some cases it could be useful to keep track of some data in a dictionary that the user can continually update along the convolution. The most relevant thing that comes to mind is a simple one-zone galactic-chemical evolution model where starformation in a closed box can change the general composition of the gas in that box. In that case, one would want to track the composition of stars that form, and how much of that is locked away in compact objects or un-evolving stars, and what the composition is of the material that is returned to the reservoir. Over time, the overall abundance of the reservoir changes.

To be able to do this, we need to keep track of data.

We can do this by utilising the `persistent_data` dictionary that is available in the `post_convolution_hook` when we perform a sequential convolution (as opposed to multiprocessing).

do note: this use-case is really just a toy model to provide some rudimentary understanding

Let us first load some relevant data. We'll use a dataset containing chemical yield of isotopes of a population of stars. 

In [2]:
import json
import pkg_resources
import numpy as np

from syntheticstellarpopconvolve.ensemble_utils import convert_ensemble_to_dataframe

# load the data
example_ensemble_filename = pkg_resources.resource_filename(
    "syntheticstellarpopconvolve", "example_data/example_ensemble.json"
)
with open(example_ensemble_filename, "r") as f_ensemble:
    ensemble = json.loads(f_ensemble.read())

This data is stored in the ensemble format, but we will inflate it, convert the isotopes to elements, and select the top 10 of elements.

First lets load the data

In [20]:
inflated_ensemble = convert_ensemble_to_dataframe(
    ensemble_data=ensemble["ensemble"]['Xyield'],
    verbose=False,
    contains_named_layers=True,
)
inflated_ensemble = inflated_ensemble.astype({'time': 'float', 'probability': 'float'})

Now lets select the unique isotopes and group them by their elements.

In [28]:
import pandas as pd
unique_isotopes = inflated_ensemble['isotope'].unique()

# remove numbers
cleaned_isotopes = [''.join([i for i in isotope if not i.isdigit()]) for isotope in unique_isotopes] 
unique_elements = np.unique(np.array(cleaned_isotopes))
print(unique_elements)

# sort on order and reverse to perform groupby
unique_elements = sorted(unique_elements, key=len)[::-1]
print(unique_elements)

#
reduced_inflated_ensemble = inflated_ensemble.copy(deep=True) 
element_df = pd.DataFrame()

# Loop over all elements
for unique_element_i, unique_element in enumerate(unique_elements):
    query = 'isotope.str.startswith("{}")'.format(unique_element)
    # print(query)

    # 
    elements = reduced_inflated_ensemble.query(query)
    elements_mask = reduced_inflated_ensemble.eval(query).to_numpy()
    # print(len(reduced_inflated_ensemble.index))

    # 
    reduced_inflated_ensemble = reduced_inflated_ensemble[np.invert(elements_mask)]
    
    # print(len(reduced_inflated_ensemble.index))
    elements = elements[['time', 'source', 'probability']]
    grouped = elements.groupby(['time', 'source']).agg({'probability': 'sum'}).reset_index()

    element_df = pd.concat([element_df, grouped])

    # print(element_df)
    # print(elements)
    # print(grouped)

    # break

print(element_df)
    
# select which elements have the largest yield (regardless of channel or delay time)







['Ag' 'Al' 'Ar' 'As' 'Au' 'B' 'Ba' 'Be' 'Bi' 'Br' 'C' 'Ca' 'Cd' 'Ce' 'Cl'
 'Co' 'Cr' 'Cs' 'Cu' 'Dy' 'Er' 'Eu' 'F' 'Fe' 'Ga' 'Gd' 'Ge' 'H' 'He' 'Hf'
 'Hg' 'Ho' 'I' 'In' 'Ir' 'K' 'Kr' 'La' 'Li' 'Lu' 'Mg' 'Mn' 'Mo' 'N' 'Na'
 'Nb' 'Nd' 'Ne' 'Ni' 'O' 'Os' 'P' 'Pb' 'Pd' 'Pm' 'Po' 'Pr' 'Pt' 'Rb' 'Re'
 'Rh' 'Ru' 'S' 'Sb' 'Sc' 'Se' 'Si' 'Sm' 'Sn' 'Sr' 'Ta' 'Tb' 'Tc' 'Te' 'Ti'
 'Tl' 'Tm' 'V' 'W' 'Xe' 'Y' 'Yb' 'Zn' 'Zr']
['Zr', 'Zn', 'Yb', 'Xe', 'Tm', 'Tl', 'Ti', 'Te', 'Tc', 'Tb', 'Ta', 'Sr', 'Sn', 'Sm', 'Si', 'Se', 'Sc', 'Sb', 'Ru', 'Rh', 'Re', 'Rb', 'Pt', 'Pr', 'Po', 'Pm', 'Pd', 'Pb', 'Os', 'Ni', 'Ne', 'Nd', 'Nb', 'Na', 'Mo', 'Mn', 'Mg', 'Lu', 'Li', 'La', 'Kr', 'Ir', 'In', 'Ho', 'Hg', 'Hf', 'He', 'Ge', 'Gd', 'Ga', 'Fe', 'Eu', 'Er', 'Dy', 'Cu', 'Cs', 'Cr', 'Co', 'Cl', 'Ce', 'Cd', 'Ca', 'Br', 'Bi', 'Be', 'Ba', 'Au', 'As', 'Ar', 'Al', 'Ag', 'Y', 'W', 'V', 'S', 'P', 'O', 'N', 'K', 'I', 'H', 'F', 'C', 'B']
     time source   probability
0     0.6  SNIbc  1.415514e-10
1     0.7  SNIbc  9.711781e-10
2

In [2]:
import copy
import json
import os

import astropy.units as u
import h5py
import numpy as np
import pandas as pd

from syntheticstellarpopconvolve import convolve, default_convolution_config
from syntheticstellarpopconvolve.general_functions import temp_dir


records = [{"a": 2, "b": 3, "normalized_yield": 1, "time": 1}]
dummy_df = pd.DataFrame.from_records(records)
TMP_DIR = temp_dir("code", "persistent_data", clean_path=True)


def post_convolution_function(
    config,
    sfr_dict,
    data_dict,
    convolution_results,
    convolution_instruction,
    persistent_data,
    previous_convolution_results,
):
    """
    Post-convolution function to handle integrating the systems forward in time and finding those that end up in the LISA waveband.

    using local_indices to select everything and using Alexey's distance sampler to handle sampling the distances
    """

    print(persistent_data)
    print(previous_convolution_results)

    if "test" not in persistent_data:
        persistent_data["test"] = 0
    persistent_data["test"] += 1

    return convolution_results


##################
#

# create file
input_hdf5_filename = os.path.join(TMP_DIR, "input_hdf5.h5")
output_hdf5_filename = os.path.join(TMP_DIR, "output_hdf5.h5")
input_hdf5_file = h5py.File(input_hdf5_filename, "w")

# Create groups main
input_hdf5_file.create_group("input_data")
input_hdf5_file.create_group("config")

# add group for events
input_hdf5_file.create_group("input_data/events")

# Write population config to file
input_hdf5_file.create_dataset("config/population", data=json.dumps({}))

# close
input_hdf5_file.close()

# store the data frame in the hdf5file
dummy_df.to_hdf(input_hdf5_filename, key="input_data/events/example")


#
convolution_config = copy.copy(default_convolution_config)
convolution_config["input_filename"] = input_hdf5_filename
convolution_config["output_filename"] = output_hdf5_filename
convolution_config["tmp_dir"] = TMP_DIR
convolution_config["redshift_interpolator_data_output_filename"] = os.path.join(
    TMP_DIR, "interpolator_dict.p"
)
convolution_config["multiply_by_time_binsize"] = False
convolution_config["filter_future_events"] = False
convolution_config["multiprocessing"] = False


###
# convolution instructions
convolution_config["convolution_instructions"] = [
    {
        "convolution_type": "integrate",
        "input_data_name": "example",
        "output_data_name": "example",
        "ignore_metallicity": True,
        "filter_future_events": False,
        "post_convolution_function": post_convolution_function,
        "data_column_dict": {
            # required
            "normalized_yield": "normalized_yield",
            "delay_time": {"column_name": "time", "unit": u.Myr},
        },
    },
]

#
convolution_config["time_type"] = "lookback_time"
convolution_config["convolution_lookback_time_bin_edges"] = np.arange(0, 6, 1) * u.Gyr


# construct the sfr-dict (NOTE: this uses absolute SFR, not metallicity dependent)
sfr_dict = {}
sfr_dict["lookback_time_bin_edges"] = (np.arange(0, 10, 1) * u.Gyr).to(u.yr)

#
scale = 1e-5
sfr_dict["starformation_rate_array"] = (
    scale * np.ones(sfr_dict["lookback_time_bin_edges"].shape[0] - 1) * u.Msun / u.yr
)  # example of a constant star-formation rate. this could be anything of course.

# store
convolution_config["SFR_info"] = sfr_dict

# convolve
convolve(config=convolution_config)

print("finished convolution")


# read out content and integrate until today
with h5py.File(convolution_config["output_filename"], "r") as output_hdf5_file:

    print(output_hdf5_file.keys())
    print(output_hdf5_file["output_data"].keys())
    print(output_hdf5_file["output_data/event"].keys())
    print(output_hdf5_file["output_data/event/example"].keys())
    print(output_hdf5_file["output_data/event/example/example"].keys())
    print(
        output_hdf5_file["output_data/event/example/example/convolution_results"].keys()
    )
    print(
        output_hdf5_file[
            "output_data/event/example/example/convolution_results/0.5 Gyr"
        ].keys()
    )

[convolve_events.py:39 - convolve_events_by_integration_post_convolution_hook_wrapper ] 2025-01-15 17:06:34,123: Handling post-convolution function hook call for convolve-events by integration


{}
None


[convolve_populations.py:113 - store_convolution_result_entries ] 2025-01-15 17:06:34,133: Storing yield
[convolve_populations.py:113 - store_convolution_result_entries ] 2025-01-15 17:06:34,135: Storing test
[convolve_events.py:39 - convolve_events_by_integration_post_convolution_hook_wrapper ] 2025-01-15 17:06:34,138: Handling post-convolution function hook call for convolve-events by integration


{'test': 1}
{'convolution_results': {'yield': <Quantity [1.e-05] 1 / yr>}}


[convolve_populations.py:113 - store_convolution_result_entries ] 2025-01-15 17:06:34,170: Storing yield
[convolve_populations.py:113 - store_convolution_result_entries ] 2025-01-15 17:06:34,175: Storing test
[convolve_events.py:39 - convolve_events_by_integration_post_convolution_hook_wrapper ] 2025-01-15 17:06:34,183: Handling post-convolution function hook call for convolve-events by integration


...

{'test': 3}
{'convolution_results': {'yield': <Quantity [1.e-05] 1 / yr>}}


[convolve_populations.py:113 - store_convolution_result_entries ] 2025-01-15 17:06:34,247: Storing yield
[convolve_populations.py:113 - store_convolution_result_entries ] 2025-01-15 17:06:34,251: Storing test
[convolve_events.py:39 - convolve_events_by_integration_post_convolution_hook_wrapper ] 2025-01-15 17:06:34,260: Handling post-convolution function hook call for convolve-events by integration


{'test': 4}
{'convolution_results': {'yield': <Quantity [1.e-05] 1 / yr>}}


[convolve_populations.py:113 - store_convolution_result_entries ] 2025-01-15 17:06:34,280: Storing yield
[convolve_populations.py:113 - store_convolution_result_entries ] 2025-01-15 17:06:34,282: Storing test


finished convolution
<KeysViewHDF5 ['config', 'input_data', 'output_data']>
<KeysViewHDF5 ['event']>
<KeysViewHDF5 ['example']>
<KeysViewHDF5 ['example']>
<KeysViewHDF5 ['convolution_results']>
<KeysViewHDF5 ['0.5 Gyr', '1.5 Gyr', '2.5 Gyr', '3.5 Gyr', '4.5 Gyr']>
<KeysViewHDF5 ['test', 'yield']>


## Further ideas
This example is a rather simple setup. Its a single box without any inflow and outflow, stars form at a fixed metallicity and they form following a prescribed star formation rate. 

Real life, of course, is somewhat more complicated. Among other processes, the ejecta and outflows of the stars will impact the metallicity and rate at which the next set of stars form. Inflow and outflow will change the reservoir abundance as well. 